### 멀티모달 RAG
- 01.multimodal 에서 추출한 텍스트와 이미지요약본을 VectorDB에 임베딩하고 

    사용자의 질문에 맞춰 관련 컨텍스트를 검색(Retrival)한뒤 LLM을 이용해서 최종 답변을 생성(Generation)

In [1]:
import os
import fitz
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

In [2]:
# 환경변수 로드 및 openai 클라이언트 생성
load_dotenv(override=True)
client = OpenAI()

In [3]:
# 벡터 Db 초기화
chroma_client = chromadb.PersistentClient(path='./chroma_db')
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv('OPENAI_API_KEY'),
    model_name = 'text-embedding-3-small'
)
collection = chroma_client.get_or_create_collection(name='multimodal_rag',embedding_function=openai_ef)
print(f'설정완료 현재 컬렉션의 크기 : {collection.count()}')

설정완료 현재 컬렉션의 크기 : 0


### 데이터 임베딩(텍스트 + 이미지 요약본)
- pdf의 텍스트와 이미지 요약본을 모두 VectorDB에 삽입(동일한 벡터공간에 텍스트와 이미지 요약본이 함께 존재)

In [ ]:
import base64
if collection.count() == 0:
    pdf_path = 'sample_paper.pdf'
    doc = fitz.open(pdf_path)
    documents = []
    metatdatas = []    
    ids = []
    
    for page_num in range(min(5,len(doc))):  # 테스트용으로 5페이지만 처리        
        page = doc.load_page(page_num)
        # 텍스트 추출
        text = page.get_text()
        if text.strip():
            documents.append(text)
            metatdatas.append({'page':page_num+1,'type':'text'})
            ids.append(f'text_page_{page_num+1}')
        # 이미지 요약 후 추출
        image_list = page.get_images(full=True)
        if image_list:
              xref = image_list[0][0]
              base_image = doc.extract_image(xref)
              image_bytes = base_image['image']
              ext = base_image['ext']
              if ext.lower() == 'jb2': ext = 'jpeg'
              mime_type = f'image/{ext}' if ext.lower() != "jpg" else 'image/jpeg'
              base64_image = base64.b64encode(image_bytes).decode('utf-8')


{'width': 1520, 'height': 2239, 'ext': 'png', 'colorspace': 3, 'xres': 96, 'yres': 96, 'bpc': 8, 'size': 121244, 'image': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x05\xf0\x00\x00\x08\xbf\x08\x02\x00\x00\x00\r\x81$Y\x00\x00\x00\tpHYs\x00\x00\x0e\xc4\x00\x00\x0e\xc4\x01\x95+\x0e\x1b\x00\x01\xd9NIDATx\x9c\xec\xdd\xff\xcf\x1dW\x9d\'\xf8\xe7?q\x9c\x1f\x9c\xbf\xc3\x86\xd0^i\xb4Rc\xdc\x19\xb4\xdb\xb4CH\xcf\xa0\xb1\xf1d\xa1\xe9\x98\x0c\x88V<\xeb!\x03V\x06\x0b\x1a"C\x00\x11\x02\x81\xde\x9d\xd8"b\x93\xe6Kw -\xb0;\xc9\x8eX\x88\x93\x1e\x8d\xc4\xc8O#\xcd\x0f+\xc5#\xed\x8f\xbd\x85\xaf\xa89\xa9{\xef\xb9\xa7\xaan\xd5\xa9:\xf5z\xe9\xa3Vx\\\xf7\xd4\xa9\xbaU\xe7y\xce\xbb\xeb\xcb\xc1?\x01\x00\x00\x000+\x07\xb9;\x00\x00\x00\x00@;\x02\x1d\x00\x00\x00\x80\x99\x11\xe8\x00\x00\x00\x00\xcc\x8c@\x07\x00\x00\x00`f\x04:\x00\x00\x00\x003#\xd0\x01\x00\x00\x00\x98\x19\x81\x0e\x00\x00\x00\xc0\xcc\x08t\x00\x00`\xb3\x834\xb9\xbb\t\xc0\x12\xf9\xf5\x03\x00\x00M\x89Q\x8ed\x07\x80\\\xfc\xd6\x01\x00\x80\xff\xa1[\x94#\xd